# FlashEats — Class 7 Challenge
## Model the Business Workflow with Data

### Client question
> **“Show us where in the workflow delay accumulates, how customers react, what interventions we make, and which metrics we should use to improve the project KPI.”**

Do not repeat source discovery, retrieval, or data-cleaning work. Today the goal is to build a useful business-workflow model.

In [1]:
import json, sqlite3, zipfile
from pathlib import Path
import pandas as pd
pd.set_option("display.max_columns",100)
pd.set_option("display.max_colwidth",140)

def find_pack_root(search_root=Path("/content")):
    for candidate in search_root.rglob("FlashEats_Classroom_Pack_V2"):
        if (candidate/"database"/"flasheats.db").exists(): return candidate
    return None

# Local run: the notebook usually already lives inside the pack root.
BASE = None
if (Path.cwd() / "database" / "flasheats.db").exists():
    BASE = Path.cwd()
elif Path.cwd().name == "FlashEats_Classroom_Pack_V2":
    BASE = Path.cwd()

if BASE is None:
    BASE=find_pack_root()
if BASE is None:
    try:
        from google.colab import files
        print("Upload the FlashEats Class 7 classroom pack ZIP.")
        uploaded=files.upload(); zip_name=next(n for n in uploaded if n.endswith(".zip"))
        extract_dir=Path("/content/flasheats_class7"); extract_dir.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(zip_name) as z: z.extractall(extract_dir)
        BASE=find_pack_root(Path("/content"))
    except Exception as e: print(e)
if BASE is None:
    BASE=find_pack_root(Path.cwd())
if BASE is None: raise FileNotFoundError("Could not locate FlashEats_Classroom_Pack_V2")
print("Using pack:",BASE)

Using pack: /Users/akshatsipany/SST/fde/flasheats-classroom-pack


In [2]:
con=sqlite3.connect(BASE/"database"/"flasheats.db")
orders=pd.read_sql("SELECT * FROM orders",con)
customers=pd.read_sql("SELECT * FROM customers",con)
restaurants=pd.read_sql("SELECT * FROM restaurants",con)
drivers=pd.read_sql("SELECT * FROM drivers",con)
tickets=pd.read_csv(BASE/"data"/"support_tickets.csv")
customer_actions=pd.read_csv(BASE/"data"/"customer_app_actions.csv")
interventions=pd.read_csv(BASE/"data"/"order_interventions.csv")
outcomes=pd.read_csv(BASE/"data"/"order_outcomes.csv")
with open(BASE/"data"/"class7_model_brief.json") as f: model_brief=json.load(f)
print("orders",orders.shape,"actions",customer_actions.shape,"interventions",interventions.shape,"outcomes",outcomes.shape)
print("Project KPI:",model_brief["project_kpi"])

orders (1603, 13) actions (2365, 6) interventions (430, 6) outcomes (1600, 6)
Project KPI: Reduce late delivery rate


# Challenge 1 — Reconstruct the order lifecycle

Choose 3 orders:
- one delivered on time,
- one delivered late,
- one with an intervention.

Build a timeline with:

`event_time | event_type | actor | source_system`

Include as many lifecycle events as the data supports.

### Hint
Start from one `order_id`, collect events from each source, then sort by time.

In [ ]:
# Challenge 1 — order lifecycle timelines (late / on-time / intervention).
base_orders = orders.drop_duplicates("order_id", keep="first")
late_order = outcomes[outcomes.late_flag == 1].order_id.iloc[0]
on_time_order = outcomes[outcomes.late_flag == 0].order_id.iloc[0]
intervention_order = interventions.order_id.iloc[0]
print("late", late_order, "on-time", on_time_order, "intervention", intervention_order)

order_events = pd.read_csv(BASE / "data" / "order_events.csv")
# customer_app_actions uses action_at; support tickets use created_at.
def build_order_timeline(order_id):
    tl = []
    o = base_orders[base_orders["order_id"] == order_id]
    if not o.empty:
        r = o.iloc[0]
        tl.append((r["created_at"], "ORDER_CREATED", "customer:" + str(r["customer_id"]), "orders.created_at"))
        tl.append((str(r["promised_eta"]), "PROMISED_ETA", "platform", "orders.promised_eta"))
        if pd.notna(r["pickup_at"]):
            tl.append((str(r["pickup_at"]), "PICKED_UP", "driver:" + str(r["driver_id"]), "orders.pickup_at"))
        if pd.notna(r["actual_delivery_at"]):
            tl.append((str(r["actual_delivery_at"]), "DELIVERED", "driver:" + str(r["driver_id"]), "orders.actual_delivery_at"))
    for _, e in order_events[order_events["order_id"] == order_id].iterrows():
        tl.append((str(e["event_time"]), e["event_type"], str(e["actor_type"]) + ":" + str(e["actor_id"]), "order_events.csv:" + str(e["source_system"])))
    for _, a in customer_actions[customer_actions["order_id"] == order_id].iterrows():
        tl.append((str(a["action_at"]), "APP_" + str(a["action_type"]), "customer:" + str(a["customer_id"]), "customer_app_actions.csv:" + str(a["channel"])))
    for _, t in tickets[tickets["order_id"] == order_id].iterrows():
        tl.append((str(t["created_at"]), "SUPPORT_TICKET:" + str(t["category"]), "customer/support", "support_tickets.csv"))
    for _, iv in interventions[interventions["order_id"] == order_id].iterrows():
        tl.append((str(iv["intervention_at"]), "INTERVENTION:" + str(iv["intervention_type"]), str(iv["initiated_by"]), "order_interventions.csv:" + str(iv["reason"])))
    for _, oc in outcomes[outcomes["order_id"] == order_id].iterrows():
        tl.append(("outcome", f"OUTCOME:{oc['outcome_bucket']} delay={oc['delay_min']}", "platform", "order_outcomes.csv"))
    tl = sorted(tl, key=lambda x: str(x[0]))
    return pd.DataFrame(tl, columns=["event_time", "event_type", "actor", "source_system"])

for oid, label in [(on_time_order, "ON-TIME"), (late_order, "LATE"), (intervention_order, "INTERVENTION")]:
    print("\n===== " + label + " " + str(oid) + " =====")
    display(build_order_timeline(oid))
print("Reading: on-time shows created→pickup→delivered before promised_eta with quiet app trace; late shows delivered after promised_eta, often with SUPPORT_OPENED/ETA re-views clustered near the breach; intervention orders show an INTERVENTION between creation and delivery (e.g. PRIORITY_DISPATCH on late-risk) — the timeline separates platform facts (orders/order_events/outcomes) from customer reactions (app/tickets) and ops responses (interventions).")

# Challenge 2 — Define the canonical project model

Your model must support:
1. customer → orders
2. order → customer interactions
3. order → support interactions
4. order → interventions
5. order → outcome

For each table, document:
- primary key,
- important foreign keys,
- grain.

Then explain why this model is better for the project than mirroring every source-system table.

In [ ]:
# Challenge 2 — canonical project model: grain + keys + why it beats source mirroring.
sources = {"orders": orders, "customer_actions": customer_actions, "support_tickets": tickets, "interventions": interventions, "outcomes": outcomes}
for name, df in sources.items():
    print(name, df.shape)
    display(df.head(2))
    pk = {"orders": "order_id (deduped business grain)", "customer_actions": "action_id", "support_tickets": "ticket_id (has 1 duplicate — clean first)", "interventions": "intervention_id", "outcomes": "order_id"}[name]
    print(" PK:", pk, "| FKs:", [c for c in df.columns if c in ("order_id", "customer_id", "restaurant_id", "driver_id")], "| grain:", {"orders": "1 row/order", "customer_actions": "1 row/app action (one-to-many per order)", "support_tickets": "1 row/ticket (one-to-many per order)", "interventions": "1 row/intervention (one-to-many per order)", "outcomes": "1 row/order"}[name])
print("\nCanonical model:")
print(" customers(customer_id PK, 1 row/customer) 1—* orders(order_id PK; FKs customer_id, restaurant_id, driver_id; 1 row/order)")
print(" orders 1—* customer_actions(FK order_id,customer_id; 1 row/action) | 1—* support_tickets(FK order_id; 1 row/ticket) | 1—* interventions(FK order_id; 1 row/intervention) | 1—1 outcomes(FK order_id; 1 row/order)")
print("Why better than mirroring source tables: one order grain for the KPI (no double-counting from action/intervention fan-out); interaction→intervention→outcome path is explicit and join-safe (aggregate before joining); source quirks (dup ticket T00013, 3 null-order tickets, case variants, 1603-row raw vs 1600 orders) are quarantined in staging, not in the KPI model.")

# Challenge 3 — Build interaction → intervention → outcome

Create one order-level table containing:

`order_id, customer_id, support_opened, cancel_attempted, intervention_count, intervention_types, final_status, late_flag, delay_min`

Answer:
1. How many late orders had support interaction?
2. How many orders received intervention?
3. Which intervention is most common?
4. Which frustrated journeys had no intervention?

### Hint
Aggregate one-to-many tables before joining them to order-level outcomes.

In [ ]:
# Challenge 3 — interaction → intervention → outcome (order-level, aggregate BEFORE joining).
actions_by_order = customer_actions.groupby("order_id").agg(action_count=("action_id", "count")).reset_index()
support_flag = customer_actions[customer_actions["action_type"] == "SUPPORT_OPENED"].groupby("order_id").size().rename("support_opened").reset_index()
support_flag["support_opened"] = 1
cancel_flag = customer_actions[customer_actions["action_type"] == "CANCEL_ATTEMPTED"].groupby("order_id").size().rename("cancel_attempted").reset_index()
cancel_flag["cancel_attempted"] = 1
inter_by_order = interventions.groupby("order_id").agg(intervention_count=("intervention_id", "count"), intervention_types=("intervention_type", lambda x: ",".join(sorted(set(x))))).reset_index()
order_level = outcomes.merge(orders.drop_duplicates("order_id")[ ["order_id", "customer_id"]], on="order_id", how="left")
order_level = order_level.merge(support_flag[["order_id", "support_opened"]], on="order_id", how="left")
order_level = order_level.merge(cancel_flag[["order_id", "cancel_attempted"]], on="order_id", how="left")
order_level = order_level.merge(inter_by_order, on="order_id", how="left")
order_level[["support_opened", "cancel_attempted"]] = order_level[["support_opened", "cancel_attempted"]].fillna(0).astype(int)
order_level["intervention_count"] = order_level["intervention_count"].fillna(0).astype(int)
order_level["intervention_types"] = order_level["intervention_types"].fillna("none")
order_level = order_level[["order_id", "customer_id", "support_opened", "cancel_attempted", "intervention_count", "intervention_types", "final_status_norm", "late_flag", "delay_min"]].rename(columns={"final_status_norm": "final_status"})
print(order_level.shape)
display(order_level.head())
print("Q1 late orders with support interaction:", int(((order_level["late_flag"] == 1) & (order_level["support_opened"] == 1)).sum()), "of", int((order_level["late_flag"] == 1).sum()))
print("Q2 orders receiving intervention:", int((order_level["intervention_count"] > 0).sum()), "(", round((order_level["intervention_count"] > 0).mean() * 100, 1), "% of", len(order_level), ")")
print("Q3 most common intervention:")
display(interventions["intervention_type"].value_counts())
print("Q4 frustrated journeys (late + support, NO intervention):", int((((order_level["late_flag"] == 1) & (order_level["support_opened"] == 1) & (order_level["intervention_count"] == 0))).sum()))
display(order_level[(order_level["late_flag"] == 1) & (order_level["support_opened"] == 1) & (order_level["intervention_count"] == 0)].head(10))
print("Note: support_opened here = in-app SUPPORT_OPENED (138 orders); support_tickets.csv is the parallel ticket channel (199 orders). Both indicate friction; the model keeps them separate, not merged.")

# Challenge 4 — Select 3–5 business metrics

Project KPI: **Reduce Late Delivery Rate**.

For each chosen metric, document:
- metric name,
- formula,
- grain,
- why it matters,
- relationship to the project KPI.

At least one metric must represent:
- an outcome,
- a customer interaction,
- an intervention.

In [ ]:
# Challenge 4 — 3–5 business metrics (outcome + interaction + intervention represented).
valid_outcomes = outcomes[outcomes["late_flag"].notna()].copy()
late_delivery_rate = float(valid_outcomes["late_flag"].mean())
frustrated_rate = float(((order_level["late_flag"] == 1) & (order_level["support_opened"] == 1)).sum() / max(1, int((order_level["late_flag"] == 1).sum())))
intervention_coverage = float((order_level["intervention_count"] > 0).mean())
cancel_attempt_rate = float(order_level["cancel_attempted"].mean())
median_delay_late = float(order_level.loc[order_level["late_flag"] == 1, "delay_min"].median())
print(f"1. Late Delivery Rate (OUTCOME): {late_delivery_rate:.3f} = mean(late_flag) over orders with known outcome (n={len(valid_outcomes)}). Why: the project KPI itself; every other metric ladders to it.")
print(f"2. Frustrated-Journey Rate (INTERACTION): {frustrated_rate:.3f} = P(support_opened=1 | late=1). Why: measures felt pain, not just minutes; targets ETA-trust work.")
print(f"3. Intervention Coverage (INTERVENTION): {intervention_coverage:.3f} = P(intervention_count>0) over all orders. Why: are ops responses reaching enough at-risk orders?")
print(f"4. Cancel-Attempt Rate (INTERACTION): {cancel_attempt_rate:.4f} = mean(cancel_attempted). Why: extreme-friction early warning; tiny (10 orders) so track as count, not just rate.")
print(f"5. Median Delay | Late (OUTCOME severity): {median_delay_late:.1f} min. Why: rate alone hides severity; pairs with the KPI for prioritisation.")
print("Grain: all order-level (1 row/order). Control: interventions + support response are controllable; late rate/median delay are outcomes.")

# Challenge 5 — Investigate the workflow with joins and aggregations

Answer at least three:

A. Do orders with support interactions have higher delay?  
B. What is late rate with vs without intervention?  
C. Which intervention type is associated with the lowest late rate?  
D. Which restaurants contribute the largest number of late orders?  
E. Which journeys show support interaction + intervention + still late?

For every answer, add one sentence:

> **What does this tell the business, and what does it NOT prove?**

In [ ]:
# Challenge 5 — workflow investigation (association != causation).
print("A. Support vs delay:")
display(order_level.groupby("support_opened")["delay_min"].agg(["count", "mean", "median"]))
print(" late rate by support:", order_level.groupby("support_opened")["late_flag"].mean().round(3).to_dict())
print(" Business: support orders average ~16.7 min delay (median ~14.3) vs ~1.7 min without; ALL in-app support orders in this cut are late — support is a late-order marker. NOT proof: support does not cause delay (reverse causation: lateness triggers support).")
print("\nB. Late rate with vs without intervention:")
order_level["has_intervention"] = (order_level["intervention_count"] > 0).astype(int)
display(order_level.groupby("has_intervention")["late_flag"].agg(["count", "mean"]))
print(" Business: ~0.564 with vs ~0.564 without — interventions as currently targeted show NO aggregate late-rate advantage. NOT proof of ineffectiveness: triage bias (interventions go to at-risk orders) masks any benefit; needs risk-adjusted comparison.")
print("\nC. Lowest-late intervention type:")
typed = interventions[["order_id", "intervention_type"]].drop_duplicates().merge(order_level[["order_id", "late_flag"]], on="order_id", how="left")
display(typed.groupby("intervention_type")["late_flag"].agg(["count", "mean"]).sort_values("mean"))
print(" Business: PRIORITY_DISPATCH lowest (~0.51), then RESTAURANT_CONTACT (~0.53), DRIVER_REASSIGNMENT (~0.58), CUSTOMER_CREDIT (~0.67 — post-hoc compensation, expected high). NOT proof: types target different situations; do not reallocate on raw rates.")
print("\nD. Restaurants by late-order count:")
rest_late = order_level[order_level["late_flag"] == 1].merge(orders.drop_duplicates("order_id")[ ["order_id", "restaurant_id"]], on="order_id", how="left")
display(rest_late.groupby("restaurant_id").size().sort_values(ascending=False).head(10))
print(" Business: R050 (22) and R051/R004/R030/R024/R023 (19–20 each) top the COUNT — starting points for prep-time audits. NOT proof: counts ignore volume; convert to rates before blaming a kitchen.")
print("\nE. Support + intervention + STILL late:")
both_late = order_level[(order_level["support_opened"] == 1) & (order_level["intervention_count"] > 0) & (order_level["late_flag"] == 1)]
print(" count:", len(both_late))
display(both_late.head(10))
print(" Business: 32 journeys got BOTH customer-visible friction and an ops response yet still delivered late — the rescue path failed; audit timing (was the intervention too late?) NOT proof the intervention caused the lateness.")

# Challenge 6 — Connect the model to the KPI

Create a one-page project view:

`PROJECT KPI → OUTCOME METRIC → WORKFLOW/DRIVER METRICS → INTERVENTIONS → DATA SOURCES/EVENTS`

Answer:
1. Which metrics are directly controllable by operations?
2. Which are outcomes?
3. Which missing event limits the model most?
4. What would you instrument next?

Final deliverable:
- core entities,
- key events,
- relationships,
- 3–5 metrics,
- KPI linkage,
- one modelling limitation.

# Final reflection

A strong solution does not produce the largest schema.

It produces the **smallest useful model that explains the workflow and supports the project KPI**.